# 13 — Advanced Quantitative Analysis

Notebook này phân tích tương quan nâng cao, momentums và beta của rổ tài sản.

In [1]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [2]:
# Cell: Import Advanced Quant Modules
import pandas as pd

from src.analytics.advanced import (
    add_cross_sectional_momentum_rank,
    compute_rolling_beta,
    correlations_by_regime,
    pair_divergence_zscore,
    screen_pair_divergences,
)
from src.analytics.correlation import build_return_matrix, compute_correlation_matrix, rolling_pair_correlation, top_correlation_pairs
from src.data.loaders import read_parquet
from src.reporting.chart_factory import (
    beta_chart,
    correlation_heatmap,
    momentum_ranking_chart,
    pair_divergence_chart,
    rolling_correlation_chart,
)


In [3]:
# Cell: Load Analysis Frame
# CUSTOMIZABLE: Define your custom watchlist here to run advanced quant and correlation diagnostics, or leave as None
user_watchlist = None  # e.g., ["SSI", "HPG", "FPT", "VIC", "VNM", "TCB", "MBB", "MWG", "DGC", "PNJ"]
analysis_df = read_parquet("data/signals/setup_signals.parquet")
volatility_df = read_parquet("data/processed/stock_features_volatility.parquet")
distribution_df = read_parquet("data/processed/stock_features_distribution.parquet")
if analysis_df.empty:
    analysis_df = volatility_df.copy()
if analysis_df.empty:
    analysis_df = distribution_df.copy()
if analysis_df.empty:
    raise FileNotFoundError(
        "Khong tim thay input cho notebook 10. Hay tao truoc setup_signals.parquet hoac stock_features_volatility.parquet"
    )
if "date" not in analysis_df.columns:
    raise KeyError(f"analysis_df khong co cot date. Current columns: {analysis_df.columns.tolist()}")
if "symbol" not in analysis_df.columns:
    raise KeyError(f"analysis_df khong co cot symbol. Current columns: {analysis_df.columns.tolist()}")
analysis_df["date"] = pd.to_datetime(analysis_df["date"])
for enrich_df, enrich_cols in [
    (volatility_df, ["volatility_regime", "volatility_percentile", "atr_pct_14"]),
    (distribution_df, ["dist_skew_60d", "dist_kurt_60d", "tail_ratio_60d"]),
]:
    if enrich_df.empty:
        continue
    enrich_df = enrich_df.copy()
    enrich_df["date"] = pd.to_datetime(enrich_df["date"])
    missing_cols = [col for col in enrich_cols if col in enrich_df.columns and col not in analysis_df.columns]
    if missing_cols:
        analysis_df = analysis_df.merge(
            enrich_df[["symbol", "date", *missing_cols]],
            on=["symbol", "date"],
            how="left",
        )
if user_watchlist is not None:
    watchlist_upper = [sym.upper() for sym in user_watchlist]
    analysis_df = analysis_df[analysis_df["symbol"].str.upper().isin(watchlist_upper)].copy()
    print(f"Using user custom watchlist: {watchlist_upper}")
analysis_df = analysis_df.sort_values(["symbol", "date"]).copy()
print("Loaded analysis_df columns:", analysis_df.columns.tolist())
analysis_df.shape

Loaded analysis_df columns: ['symbol', 'market', 'date', 'open', 'high', 'low', 'close', 'volume', 'value', 'return_1d', 'return_5d', 'forward_return_5d', 'return_10d', 'forward_return_10d', 'return_20d', 'forward_return_20d', 'return_60d', 'forward_return_60d', 'log_return_1d', 'ma20', 'distance_ma20', 'above_ma20', 'ma50', 'distance_ma50', 'above_ma50', 'ma200', 'distance_ma200', 'above_ma200', 'true_range', 'atr_14', 'atr_pct_14', 'realized_vol_10d', 'realized_vol_20d', 'realized_vol_60d', 'ewma_vol_20d', 'volatility_percentile', 'volatility_regime', 'dist_mean_60d', 'dist_median_60d', 'dist_skew_60d', 'dist_kurt_60d', 'dist_q05_60d', 'dist_q95_60d', 'tail_ratio_60d', 'volume_ma20', 'volume_zscore_20d', 'value_ma20', 'value_zscore_20d', 'breakout_20d', 'pullback_ma20', 'volume_spike']


(186827, 51)

In [4]:
# Cell: Build Correlation Views
return_matrix = build_return_matrix(analysis_df, return_col="return_1d")
corr_matrix = compute_correlation_matrix(return_matrix, min_periods=40)
top_pairs = top_correlation_pairs(corr_matrix, top_n=15, min_abs_correlation=0.4)
display(top_pairs)
correlation_heatmap(corr_matrix, title="Watchlist Return Correlation Heatmap")


,symbol_a,symbol_b,correlation,abs_correlation
0,ATS,TTE,-1.000000,1.000000
1,CFPT2525,CFPT2527,0.891841,0.891841
2,CFPT2527,FPT,0.889240,0.889240
3,MSB,OCB,0.878213,0.878213
4,CACB2514,CACB2515,0.869879,0.869879
5,ACB,CACB2515,0.866326,0.866326
6,CFPT2527,CFPT2529,0.864858,0.864858
7,ACB,CACB2605,0.864294,0.864294
8,CFPT2529,FPT,0.863637,0.863637
9,BID,VCB,0.854811,0.854811


In [5]:
# Cell: Rolling Correlation And Divergence
pair = top_pairs.iloc[0] if not top_pairs.empty else pd.Series({"symbol_a": "SSI", "symbol_b": "HPG"})

rolling_corr = rolling_pair_correlation(
    analysis_df,
    symbol_a=pair["symbol_a"],
    symbol_b=pair["symbol_b"],
    window=60,
)
divergence = pair_divergence_zscore(
    analysis_df,
    symbol_a=pair["symbol_a"],
    symbol_b=pair["symbol_b"],
    window=60,
)

display(rolling_corr.tail())
display(divergence.tail())
display(rolling_correlation_chart(rolling_corr, title="60-day Rolling Correlation"))
pair_divergence_chart(divergence, title=f"Pair Divergence: {pair['symbol_a']} vs {pair['symbol_b']}")


,date,symbol_a,symbol_b,rolling_correlation
130,2026-06-19,ATS,TTE,NaN
131,2026-06-22,ATS,TTE,NaN
132,2026-06-23,ATS,TTE,NaN
133,2026-06-24,ATS,TTE,NaN
134,2026-06-25,ATS,TTE,NaN


,date,symbol_a,symbol_b,price_a,price_b,spread,rolling_mean,rolling_std,divergence_zscore
37,2026-06-18,ATS,TTE,27400.0,33900.0,-0.212872,NaN,NaN,NaN
38,2026-06-19,ATS,TTE,27400.0,33900.0,-0.212872,NaN,NaN,NaN
39,2026-06-22,ATS,TTE,27400.0,33900.0,-0.212872,NaN,NaN,NaN
40,2026-06-23,ATS,TTE,27400.0,33900.0,-0.212872,NaN,NaN,NaN
41,2026-06-24,ATS,TTE,27400.0,33900.0,-0.212872,NaN,NaN,NaN


In [6]:
# Cell: Screen Pair Divergences
divergence_screen = screen_pair_divergences(
    analysis_df,
    corr_matrix=corr_matrix,
    top_n_pairs=10,
    min_abs_correlation=0.4,
    zscore_window=60,
)
divergence_screen


,date,symbol_a,symbol_b,price_a,price_b,spread,rolling_mean,rolling_std,divergence_zscore,correlation,abs_correlation
0,2026-03-05,CFPT2527,CFPT2529,10.0,2420.0,-5.488938,-1.132637,0.906472,-4.805778,0.864858,0.864858
1,2026-03-05,CFPT2525,CFPT2527,70.0,10.0,1.945910,-0.793319,0.570069,4.805082,0.891841,0.891841
2,2026-03-05,CFPT2527,FPT,10.0,83252.0,-9.027042,-4.342671,0.995020,-4.707814,0.889240,0.889240
3,2026-06-23,CFPT2529,FPT,680.0,70800.0,-4.645521,-4.350346,0.150843,-1.956837,0.863637,0.863637
4,2026-06-26,MSB,OCB,15800.0,12550.0,0.230289,0.161997,0.072352,0.943885,0.878213,0.878213
5,2026-06-23,CACB2514,CACB2515,690.0,1300.0,-0.633428,-0.563954,0.119688,-0.580460,0.869879,0.869879
6,2026-06-22,ACB,CACB2605,22000.0,1330.0,2.805864,2.911331,0.231522,-0.455537,0.864294,0.864294
7,2026-06-23,ACB,CACB2515,22350.0,1300.0,2.844462,2.698694,0.332362,0.438582,0.866326,0.866326
8,2026-06-26,BID,VCB,41850.0,61200.0,-0.380055,-0.383370,0.016236,0.204186,0.854811,0.854811


In [7]:
# Cell: Cross Sectional Momentum Ranking
momentum_df = add_cross_sectional_momentum_rank(analysis_df, lookback=20)
latest_momentum = momentum_df[momentum_df["date"] == momentum_df["date"].max()].copy()
display(latest_momentum.sort_values("momentum_rank_20d").head(10))
momentum_ranking_chart(latest_momentum, momentum_col="momentum_20d", top_n=10, title="Top 10 Momentum Names")


,symbol,market,date,open,high,low,close,volume,value,return_1d,...,volume_zscore_20d,value_ma20,value_zscore_20d,breakout_20d,pullback_ma20,volume_spike,momentum_20d,momentum_pct_rank_20d,momentum_rank_20d,momentum_decile_20d
85754,HSL,HOSE,2026-06-26,7300,7450.0,7190.0,7190,224500.0,1.630144e+09,-0.028378,...,-0.591352,6.087902e+09,-0.564752,False,False,False,0.491701,1.000000,1.0,10.0
154319,THD,HNX,2026-06-26,172000,178900.0,164000.0,178900,0.0,0.000000e+00,0.040116,...,-0.873253,7.814418e+09,-0.971105,False,False,False,0.463993,0.995370,2.0,10.0
42570,CLW,HOSE,2026-06-26,63000,63000.0,63000.0,63000,100.0,6.300000e+06,0.065990,...,-0.241843,3.184875e+08,-0.237023,False,False,False,0.361367,0.990741,3.0,10.0
115178,NVB,HNX,2026-06-26,14700,14900.0,14400.0,14800,0.0,0.000000e+00,0.013699,...,-1.041816,7.246211e+09,-0.968489,False,False,False,0.298246,0.986111,4.0,10.0
99022,LDG,HOSE,2026-06-26,3550,3700.0,3510.0,3550,618200.0,2.231764e+09,0.014286,...,-0.704058,5.612184e+09,-0.656797,False,False,False,0.254417,0.981481,5.0,10.0
97998,LAF,HOSE,2026-06-26,20750,20750.0,20750.0,20750,128700.0,2.670525e+09,0.069588,...,3.748373,3.053020e+08,3.878633,True,False,True,0.224189,0.976852,6.0,10.0
186262,VVS,HOSE,2026-06-26,84400,86000.0,83900.0,84800,43400.0,3.695680e+09,0.001181,...,-1.476712,1.469949e+10,-1.612293,False,False,False,0.219828,0.972222,7.0,10.0
161915,TVC,HNX,2026-06-26,10500,10900.0,10500.0,10600,0.0,0.000000e+00,0.009524,...,-0.789754,4.156137e+09,-0.719617,False,False,False,0.218391,0.967593,8.0,10.0
65148,EVS,HNX,2026-06-26,5500,5500.0,5400.0,5400,0.0,0.000000e+00,0.000000,...,-0.996648,9.829590e+08,-0.931150,False,False,False,0.200000,0.962963,9.0,10.0
135070,SBG,HOSE,2026-06-26,12500,12550.0,12400.0,12550,115300.0,1.440515e+09,0.000000,...,-1.447073,3.816840e+09,-1.390352,False,False,False,0.185864,0.958333,10.0,10.0


In [8]:
# Cell: Rolling Beta Vs VNINDEX
vnindex_path = Path("data/raw/index_ohlcv/VNINDEX.parquet")
if vnindex_path.exists():
    vnindex = read_parquet(vnindex_path)
    vnindex["date"] = pd.to_datetime(vnindex["date"])
    beta_df = compute_rolling_beta(
        analysis_df,
        benchmark_df=vnindex,
        benchmark_symbol="VNINDEX",
        window=60,
    )
    display(beta_df[["symbol", "date", "rolling_beta_60d"]].dropna().tail(10))
    beta_chart(beta_df, symbol=beta_df["symbol"].iloc[0], beta_col="rolling_beta_60d", title="Rolling Beta vs VNINDEX")
else:
    pd.DataFrame({"info": ["Chua co data/raw/index_ohlcv/VNINDEX.parquet. Them benchmark data de bat rolling beta vs VNINDEX."]})


,symbol,date,rolling_beta_60d
186815,YEG,2026-06-11,0.677445
186816,YEG,2026-06-12,0.655489
186817,YEG,2026-06-15,0.661899
186818,YEG,2026-06-16,0.665111
186819,YEG,2026-06-17,0.677618
186820,YEG,2026-06-18,0.432662
186821,YEG,2026-06-19,0.438682
186822,YEG,2026-06-22,0.385839
186823,YEG,2026-06-23,0.402815
186824,YEG,2026-06-24,0.402049


In [9]:
# Cell: Correlation By Regime
regime_corr = correlations_by_regime(
    analysis_df,
    regimes=["Low Vol", "Normal Vol", "Panic Vol"],
    regime_col="volatility_regime",
    min_periods=20,
)

selected_regime = "Low Vol"
display(regime_corr[selected_regime])
correlation_heatmap(regime_corr[selected_regime], title=f"Correlation Heatmap - {selected_regime}")


symbol,AAA,AAH,AAS,AAT,ABB,ABC,ABI,ABT,ABW,ACB,...,VPH,VPI,VPS,VRE,VSC,VSH,VTO,VTP,VTZ,YEG
symbol,,,,,,,,,,,,,,,,,,,,,
AAA,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-0.019169,0.284516,NaN,NaN,0.067698,NaN,NaN
AAH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ABB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VSH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VTP,0.067698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.265856,NaN,NaN,1.000000,NaN,NaN
